In [1]:
from ech_discovery.data.aggregates import common
from ech_discovery.notebook import *

In [2]:
df = (
    pl.read_ndjson(
        config.RAW_DATA_DIR
        / "tud"
        / "echtool/configuration-variations-*cert-noverify/*",
        include_file_paths=C.filename,
    )
    .with_columns(pl.col(C.filename).str.extract("(tcp|quic)").alias(C.role))
    .with_columns(
        pl.col(C.filename)
        .str.split("/")
        .list.get(-1)
        .str.replace_all("_", ":")
        .str.replace(".jsonl.zst", "")
        .str.to_datetime("%+")
        .dt.date()
        .alias(C.measurement)
    )
    .filter(pl.col(C.filename) == pl.col(C.filename).max().over(C.role))
    .drop(C.filename)
    .with_columns(
        # Combine certificate errors
        pl.when(
            pl.col(C.detail).str.contains(
                "certificate has expired|certificate is valid for"
            )
        )
        .then(pl.lit(EchtoolErrorKind.certificate_error))
        # QUIC error
        .when(pl.col(C.detail).str.contains("INVALID_TOKEN"))
        .then(pl.lit(EchtoolErrorKind.quic_invalid_token))
        # QUIC error
        .when(
            pl.col(C.detail).str.contains(
                "CRYPTO_ERROR\(112\)|CRYPTO_ERROR\(40\)|CRYPTO_ERROR\(80\)"
            )
        )
        .then(pl.lit(EchtoolErrorKind.tls_alert))
        .when(pl.col(C.detail) == "EOF")
        .then(pl.lit(EchtoolErrorKind.handshake_failure))
        .when(pl.col(C.status) == EchtoolErrorKind.accepted)
        .then(pl.col(C.error_kind).fill_null(EchtoolErrorKind.accepted))
        .when(
            (pl.col(C.error_kind) == EchtoolErrorKind.ech_rejected)
            & (pl.col(C.retry_config).is_not_null())
        )
        .then(pl.lit(EchtoolErrorKind.ech_rejected_with_retry))
        .otherwise(pl.col(C.error_kind))
        .alias(C.error_kind)
    )
)

In [3]:
# Get ECH public name
domains = df.select(pl.col(C.domain).unique())
domains_and_public_name = (
    common.ech_with_dns_status(
        limit_per_location=True, regex_filter="tud/2026-07-30", limit=1
    )
    .select(C.target, C.ech_public_name)
    .join(domains.lazy(), left_on=C.target, right_on=C.domain, how="semi")
    .with_columns(
        # nordvpn returns individual response domains, fix this manually in the matching dataset
        pl.when(
            (pl.col(C.target) == "au765.proxy.nordvpn.com")
            & (pl.col(C.ech_public_name) == "bChFPfc7.example.com")
        )
        .then(pl.lit("qkNsHvVS.example.com"))
        .otherwise(C.ech_public_name)
        .alias(C.ech_public_name)
    )
    .unique()
).collect()

assert (
    domains_and_public_name.group_by(C.target)
    .agg(pl.col(C.ech_public_name).n_unique())
    .filter(pl.col(C.ech_public_name) > 1)
    .select(pl.len())
    .item()
    == 0
), "Multiple ECH public names per domain"

In [4]:
# Enrich ech public name
df = df.join(
    domains_and_public_name,
    left_on=C.domain,
    right_on=C.target,
    how="left",
    validate="m:1",
)

In [5]:
# we show these as n/a, they do not carry meaning of the ECH connection
connection_errors = pl.col(C.detail).str.contains(
    "handshake timeout| DNS lookup for |dial error: timed out after | network is unreachable| i/o timeout"
) | pl.col(C.error_kind).is_in([EchtoolErrorKind.network_error.value, EchtoolErrorKind.dns_error.value])

In [6]:
TEST_CASES = 4

common_behavior = (
    df.filter(
        # Do not check no change case
        pl.col(C.original) != pl.col(C.replaced),
        ~connection_errors,
        pl.col(C.kind) != C.outer_max_name_length,
    )
    .group_by(C.role, C.ech_public_name, C.kind, C.measurement)
    .agg(pl.coalesce(C.error_kind, C.status).unique().alias(C.status))
)

coverage = (
    df.select(C.ech_public_name)
    .unique()
    .join(
        common_behavior.group_by(C.ech_public_name).agg(
            pl.col(C.kind).n_unique().alias(C.len)
        ),
        on=C.ech_public_name,
        how="left",
    )
    .with_columns(pl.col(C.len).fill_null(0))
)
# Expect all test cases to be fulfilled
incomplete = coverage.filter(pl.col(C.len) != TEST_CASES)

print(
    f"{len(incomplete)} of {len(coverage)} public names lack data for all {TEST_CASES} "
    f"test-cases, {len(coverage.filter(pl.col(C.len) == 0))} have no data at all"
)
display(incomplete.sort(C.len))
common_behavior

11 of 1363 public names lack data for all 4 test-cases, 2 have no data at all


ech_public_name,len
str,u32
"""ech.cdn.nexus""",0
"""I49B.UF4G.xPcF""",0
"""nvrxDMhH.example.com""",2
…,…
"""aRYCYLTy.example.com""",2
"""EnI3KmnZ.example.com""",2


role,ech_public_name,kind,measurement,status
str,str,str,date,list[str]
"""quic""","""hYY34iV9.example.com""","""outer_ech_config_id""",2026-08-10,"[""accepted""]"
"""tcp""","""HHS2yOLv.example.com""","""ech_config_public_name""",2026-08-10,"[""ech_rejected_with_retry""]"
"""quic""","""uwRls0Ns.example.com""","""ech_config_config_id""",2026-08-10,"[""ech_rejected_with_retry""]"
…,…,…,…,…
"""tcp""","""WPk99Kfp.example.com""","""outer_ech_config_id""",2026-08-10,"[""accepted""]"
"""quic""","""0J4l014w.example.com""","""outer_ech_config_id""",2026-08-10,"[""accepted""]"


In [7]:
# Aggregate information per_ech_public_name and also per transport parameter.
d = (
    common_behavior.group_by(C.ech_public_name, C.measurement, C.kind)
    .agg(pl.col(C.status).list.explode(empty_as_null=True).unique())
    .sort(C.kind)
    .pivot(on=C.kind, index=[C.ech_public_name, C.measurement])
)
d.write_parquet(config.PROCESSED_DATA_DIR / "echtool_configuration_variations.pq.zst")